### Il notebook e main.py eseguono sostanzialmente la stessa pipeline,
### ma hanno scopi diversi:
### main.py esegue automaticamente tutto il progetto,
### mentre pteamLab.ipynb serve per studiare e visualizzare ogni fase.


## Import delle librerie e dei moduli

Importiamo le funzioni dei vari moduli di PurpleMind.
In questo modo il notebook può utilizzare il codice del progetto
senza doverlo riscrivere.


In [ ]:
from config import (
    EVENTI_NORMALI,
    EVENTI_ATTACCO
)

from traffic import genera_traffico

from dataset import (
    crea_dataframe,
    salva_dataset,
    prepara_dati,
    dividi_dati
)

from eda import (
    analizza_dataset,
    grafico_bytes,
    grafico_correlazioni,
    grafico_pairplot
)

from models import (
    allena_logistic_regression,
    allena_random_forest,
    allena_isolation_forest,
    predici,
    predici_anomalie
)

from evaluation import (
    valuta_modello,
    crea_confronto,
    scegli_miglior_modello
)


## Generazione del traffico

Il Red Team genera il traffico simulato.

Creiamo eventi normali e eventi di attacco utilizzando Scapy.


In [ ]:
eventi = genera_traffico(
    EVENTI_NORMALI,
    EVENTI_ATTACCO
)

len(eventi)


## Creazione del dataset

Trasformiamo gli eventi generati in un DataFrame Pandas.

Successivamente salviamo il dataset in formato CSV.


In [ ]:
dataframe = crea_dataframe(eventi)

salva_dataset(dataframe)

dataframe.head()


## Controllo del dataset

Controlliamo le dimensioni del dataset e la distribuzione
tra eventi benign e malicious.

Questo ci permette di verificare che i dati siano stati generati correttamente.


In [ ]:
print("Dimensioni:", dataframe.shape)

print("\nDistribuzione:")
print(dataframe["label"].value_counts())


## Exploratory Data Analysis

Analizziamo il dataset prima del Machine Learning.

Studiamo le statistiche, la distribuzione delle feature,
le correlazioni e le differenze tra traffico normale e traffico di attacco.


In [ ]:
analizza_dataset(dataframe)

grafico_bytes(dataframe)

grafico_correlazioni(dataframe)

grafico_pairplot(dataframe)


## Preparazione dei dati per il Machine Learning

Selezioniamo le feature che verranno utilizzate dai modelli.

Convertiamo le label in valori numerici e dividiamo il dataset
in training set e test set.


In [ ]:
X, y = prepara_dati(dataframe)

X_train, X_test, y_train, y_test = dividi_dati(
    X,
    y
)

print("Training:", X_train.shape)
print("Test:", X_test.shape)


## Blue Team V1 — Logistic Regression

Utilizziamo Logistic Regression come modello di riferimento.

Il modello impara dai dati di training e successivamente
classifica gli eventi presenti nel test set.


In [ ]:
modello_logistic = allena_logistic_regression(
    X_train,
    y_train
)

predizione_logistic = predici(
    modello_logistic,
    X_test
)


## Blue Team V2 — Random Forest

Utilizziamo Random Forest come secondo modello di detection.

L'obiettivo è verificare se un modello più complesso
riesce a rilevare meglio gli attacchi.


In [ ]:
modello_random_forest = allena_random_forest(
    X_train,
    y_train
)

predizione_random_forest = predici(
    modello_random_forest,
    X_test
)


## Blue Team V3 — Isolation Forest

Utilizziamo Isolation Forest per l'Anomaly Detection.

A differenza dei modelli supervisionati, cerca principalmente
eventi anomali senza utilizzare direttamente le label durante l'addestramento.


In [ ]:
modello_isolation = allena_isolation_forest(
    X_train
)

predizione_isolation = predici_anomalie(
    modello_isolation,
    X_test
)


## Purple Team — Confronto dei modelli

Il Purple Team confronta le prestazioni dei tre Blue Team.

Analizziamo Accuracy, Precision, Recall,
False Positives e False Negatives.


In [ ]:
risultati = [
    valuta_modello(
        "Logistic Regression",
        y_test,
        predizione_logistic
    ),
    valuta_modello(
        "Random Forest",
        y_test,
        predizione_random_forest
    ),
    valuta_modello(
        "Isolation Forest",
        y_test,
        predizione_isolation
    )
]

confronto = crea_confronto(risultati)

confronto


## Purple Team Verdict

Selezioniamo il modello con il Recall più alto.

In cybersecurity il Recall è particolarmente importante
perché indica quanti attacchi reali vengono correttamente rilevati.


In [ ]:
miglior_modello = scegli_miglior_modello(
    confronto
)

print("Miglior Defender:")
print(miglior_modello["Modello"])

print("\nRecall:")
print(round(miglior_modello["Recall"], 3))
